# Random Forest from Scratch

The aim of this notebook is to implement **random forest classifier and regressor** algorithms from scratch using only NumPy. 

In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

## 1. Random Forest Classifier

In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score
from decision_tree import DecisionTreeClassifier

In [3]:
df_classification = pd.DataFrame(load_breast_cancer().data, columns=load_breast_cancer().feature_names)
df_classification['target'] = load_breast_cancer().target # binary classification: malignant or benign

In [4]:
df_classification.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [5]:
# Prepare the data: target variable and features
X_classification = df_classification.drop("target", axis=1).values # features
y_classification = df_classification["target"].values # target

In [6]:
# Split datasets into training and testing sets
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X_classification, y_classification, test_size=0.2, random_state=42)

In [7]:
# Boostrap sampling function
def boostrap_sampling(X, y):
    # Number of training exemples
    n_examples = len(X[:, 0])

    # Sample indices with replacement
    sampled_indices = np.random.choice(n_examples, n_examples, replace=True)

    # Get the sampled data
    X_sample = X[sampled_indices]
    y_sample = y[sampled_indices]

    # Get the out-of-bag (OOB) indices
    oob_indices = np.setdiff1d(np.arange(n_examples), sampled_indices) # return the indices that are not in the sampled_indices

    return X_sample, y_sample, oob_indices

In [8]:
# Test with the training set
Xc_sample, yc_sample, oob_indices_c = boostrap_sampling(Xc_train, yc_train)
print("Sampled data shape:", Xc_sample.shape)
print("Sampled target shape:", yc_sample.shape)
print("Out-of-bag indices:", oob_indices_c)
print("Out-of-bag shape:", oob_indices_c.shape)

Sampled data shape: (455, 30)
Sampled target shape: (455,)
Out-of-bag indices: [  4  13  15  18  25  26  28  29  30  31  32  35  37  41  43  44  45  46
  48  50  54  55  59  60  61  62  63  65  71  75  77  79  86  87  89  91
  93  94  96  98 100 103 107 108 109 110 111 114 115 120 121 126 127 128
 129 130 131 132 135 141 145 146 149 151 152 159 160 163 166 172 174 181
 182 185 190 198 200 202 203 206 209 211 220 221 222 226 227 228 236 239
 243 247 249 259 263 268 273 287 288 291 292 293 296 297 298 303 308 309
 310 312 313 317 318 322 325 326 327 328 329 331 335 336 341 343 355 357
 359 361 362 364 366 367 368 369 373 377 378 382 385 388 390 393 394 395
 400 402 404 406 408 412 419 420 421 422 424 429 434 439 444 445 454]
Out-of-bag shape: (161,)


Out-of-bag shape / Sampled data shape = 168 / 455 ≈ 36.9%, exactly what we expected. 

In [9]:
assert len(Xc_sample) == len(Xc_train) # len(unique sampled indices) + len(oob_indices) == n

In [10]:
# Training loop
def fit_classifier(X, y, n_estimators=10, max_features=5, max_depth=3, min_samples_split=2, min_samples_leaf=1):
    # List to store trees
    forest = []

    # Loop over each tree
    for _ in range(n_estimators):
        X_sample, y_sample, oob_indices = boostrap_sampling(X, y)
        model = DecisionTreeClassifier(max_depth=max_depth, max_features=max_features, min_samples_split=min_samples_split, min_samples_leaf=min_samples_leaf)
        model.fit(X_sample, y_sample)
        forest.append((model, oob_indices))

    return forest

In [11]:
fit_classifier(Xc_train, yc_train)

[(<decision_tree.DecisionTreeClassifier at 0x108546e70>,
  array([  3,  11,  18,  19,  20,  21,  23,  24,  26,  27,  30,  33,  35,
          38,  41,  43,  45,  47,  49,  51,  57,  59,  60,  63,  64,  65,
          68,  69,  70,  72,  73,  74,  77,  83,  84,  85,  88,  93,  99,
         101, 102, 103, 106, 115, 117, 118, 119, 123, 125, 129, 132, 133,
         134, 140, 151, 156, 158, 159, 162, 166, 169, 174, 180, 185, 186,
         191, 195, 200, 203, 206, 213, 216, 217, 219, 220, 222, 223, 226,
         227, 228, 230, 231, 235, 242, 244, 248, 252, 253, 259, 261, 263,
         264, 267, 268, 269, 271, 273, 276, 278, 285, 286, 287, 291, 293,
         296, 298, 304, 305, 313, 314, 315, 317, 318, 321, 323, 325, 326,
         327, 330, 331, 339, 340, 343, 352, 353, 355, 356, 357, 360, 370,
         372, 373, 374, 377, 378, 379, 384, 387, 392, 396, 399, 400, 401,
         403, 405, 406, 408, 409, 414, 415, 417, 420, 421, 423, 425, 426,
         427, 430, 434, 440, 444, 445, 449, 450, 451]))

In [12]:
# Prediction by majority vote
def predict_classifier(forest, X):
    # List to store predictions from each tree
    tree_predictions = []

    # Loop over each tree in the forest
    for model, _ in forest:
        tree_predictions.append(model.predict(X))

    # Stack predictions from all trees into a 2D array (matrix)
    tree_predictions = np.stack(tree_predictions, axis=1) # shape (n_samples, n_estimators)
    tree_predictions = tree_predictions.astype(int)

    # Majority vote: for each sample, find the most common prediction among the trees
    predictions = np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=1, arr=tree_predictions)

    return predictions

In [13]:
yc_pred = predict_classifier(fit_classifier(Xc_train, yc_train), Xc_test)
accuracy_classification = accuracy_score(yc_test, yc_pred)
print("Accuracy:", accuracy_classification)

Accuracy: 0.9649122807017544


In [32]:
# Compute the OOB error
def oob_score_classification(forest, X, y):
    # Initialize an empty list to store OOB predictions for each sample
    n_samples = len(X)
    oob_predictions = [[] for _ in range(n_samples)]

    # For each tree in the forest, get the OOB indices and make predictions for those samples
    for model, oob_indices in forest:
        predictions = model.predict(X[oob_indices]).astype(int)
        for idx, pred in zip(oob_indices, predictions):
            oob_predictions[idx].append(pred)

    # For each sample, find the most common prediction among the trees that did not use it for training
    final_oob_predictions = np.zeros(n_samples, dtype=int)
    for i in range(n_samples):
        # Get the predictions from trees that did not use this sample for training
        tree_preds = np.array(oob_predictions[i])
        if len(tree_preds) > 0: # if there are any OOB predictions for this sample
            final_oob_predictions[i] = np.bincount(tree_preds).argmax() # majority vote

    # Calculate OOB accuracy
    oob_accuracy = accuracy_score(y, final_oob_predictions)
    return oob_accuracy

In [33]:
oob_accuracy = oob_score_classification(fit_classifier(Xc_train, yc_train), Xc_train, yc_train)
print("OOB Accuracy:", oob_accuracy)

OOB Accuracy: 0.9274725274725275


In [34]:
def feature_importance_classification(forest, n_features):
    # Initialize an array to store feature importance scores
    importance_scores = np.zeros(n_features)

    # Recursive function to traverse the tree and update importance scores
    def _traverse(node):
        if node.value is not None:  # leaf, we stop traversing
            return
        importance_scores[node.feature] += 1
        _traverse(node.left)
        _traverse(node.right)

    # For each tree in the forest, check if the node is a leaf node or not, if not, get the feature index used for splitting and update the importance score for that feature
    for model, _ in forest:
        # Check if the node is a leaf node
        _traverse(model.tree)

    # Normalize importance scores
    importance_scores /= np.sum(importance_scores)

    return importance_scores

In [38]:
features_classification = load_breast_cancer().feature_names
importances_classification = feature_importance_classification(fit_classifier(Xc_train, yc_train), Xc_train.shape[1])

sorted_idx_classification = np.argsort(importances_classification)[::-1]
for i in sorted_idx_classification:
    print(f"{features_classification[i]}: {importances_classification[i]:.4f}")

worst area: 0.0923
worst concave points: 0.0769
mean concave points: 0.0769
mean radius: 0.0615
worst radius: 0.0615
worst symmetry: 0.0615
mean concavity: 0.0615
worst texture: 0.0615
worst perimeter: 0.0615
worst smoothness: 0.0615
mean perimeter: 0.0615
mean compactness: 0.0462
mean area: 0.0308
texture error: 0.0308
perimeter error: 0.0154
mean fractal dimension: 0.0154
mean texture: 0.0154
worst fractal dimension: 0.0154
compactness error: 0.0154
concavity error: 0.0154
concave points error: 0.0154
symmetry error: 0.0154
worst compactness: 0.0154
worst concavity: 0.0154
area error: 0.0000
radius error: 0.0000
mean symmetry: 0.0000
mean smoothness: 0.0000
fractal dimension error: 0.0000
smoothness error: 0.0000


## 2. Random Forest Regressor

In [18]:
from sklearn.datasets import load_diabetes
from sklearn.metrics import r2_score, root_mean_squared_error
from regression_tree import RegressionTree

In [19]:
# Load the diabetes dataset
df_regression = pd.DataFrame(load_diabetes().data, columns=load_diabetes().feature_names)
df_regression['target'] = load_diabetes().target # diabete progression measure after one year

In [20]:
df_regression.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [21]:
# Prepare the data: target variable and features
X_regression = df_regression.drop("target", axis=1).values # features
y_regression = df_regression["target"].values # target

In [22]:
# Split datasets into training and testing sets
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_regression, y_regression, test_size=0.2, random_state=42)

In [23]:
# Boostrap sampling with the training set
Xr_sample, yr_sample, oob_indices_r = boostrap_sampling(Xr_train, yr_train)
print("Sampled data shape:", Xr_sample.shape)
print("Sampled target shape:", yr_sample.shape)
print("Out-of-bag indices:", oob_indices_r)
print("Out-of-bag shape:", oob_indices_r.shape)

Sampled data shape: (353, 10)
Sampled target shape: (353,)
Out-of-bag indices: [ 10  17  18  23  26  27  28  29  31  32  33  39  41  42  45  46  49  53
  57  60  61  62  63  65  67  68  75  77  78  79  82  84  91  93  94  96
  97 100 101 107 112 114 117 118 119 121 126 133 138 139 140 142 143 146
 151 155 157 162 163 165 170 172 173 175 179 182 184 188 192 193 194 196
 207 209 212 216 221 224 225 226 227 229 232 233 240 243 244 245 246 247
 249 252 253 255 258 262 265 269 271 273 279 284 286 290 294 295 301 303
 307 309 310 312 313 316 317 318 324 327 329 330 331 337 338 348 351 352]
Out-of-bag shape: (126,)


In [24]:
assert len(Xr_sample) == len(Xr_train) # len(unique sampled indices) + len(oob_indices) == n

In [25]:
# Training loop
def fit_regressor(X, y, n_estimators=10, max_features=5, max_depth=3, min_samples_split=2):
    # List to store trees
    forest = []

    # Loop over each tree
    for _ in range(n_estimators):
        X_sample, y_sample, oob_indices = boostrap_sampling(X, y)
        model = RegressionTree(max_depth=max_depth, max_features=max_features, min_samples_split=min_samples_split)
        model.fit(X_sample, y_sample)
        forest.append((model, oob_indices))

    return forest

In [26]:
fit_regressor(Xr_train, yr_train)

[(<regression_tree.RegressionTree at 0x12f224950>,
  array([  0,   5,   6,  11,  13,  22,  25,  31,  33,  34,  35,  37,  40,
          44,  46,  48,  49,  51,  54,  56,  57,  61,  68,  69,  71,  77,
          81,  85,  86,  91,  95,  96,  97,  99, 100, 103, 105, 109, 111,
         112, 114, 118, 119, 120, 122, 123, 124, 125, 127, 128, 132, 138,
         141, 145, 146, 147, 148, 151, 153, 155, 160, 162, 163, 166, 168,
         177, 178, 179, 180, 186, 188, 190, 192, 193, 200, 205, 206, 208,
         210, 213, 215, 224, 226, 227, 228, 231, 235, 242, 243, 247, 248,
         249, 250, 252, 253, 254, 255, 256, 257, 262, 265, 266, 267, 270,
         275, 276, 278, 289, 290, 292, 294, 299, 301, 302, 307, 308, 310,
         311, 315, 316, 318, 322, 324, 327, 330, 334, 341, 342, 343, 344,
         346, 351, 352])),
 (<regression_tree.RegressionTree at 0x12f224260>,
  array([  8,  25,  27,  30,  33,  35,  37,  38,  44,  48,  49,  50,  51,
          52,  54,  55,  57,  58,  62,  67,  68,  70,  71

In [27]:
# Prediction by mean
def predict_regressor(forest, X):
    # List to store predictions from each tree
    tree_predictions = []

    # Loop over each tree in the forest
    for model, _ in forest:
        tree_predictions.append(model.predict(X))

    # Stack predictions from all trees into a 2D array (matrix)
    tree_predictions = np.stack(tree_predictions, axis=1) # shape (n_samples, n_estimators)

    # Mean vote
    predictions = np.apply_along_axis(lambda x: np.mean(x), axis=1, arr=tree_predictions)

    return predictions

In [28]:
yr_pred = predict_regressor(fit_regressor(Xr_train, yr_train), Xr_test)
r2_score_regression = r2_score(yr_test, yr_pred)
rmse_regression = root_mean_squared_error(yr_test, yr_pred)

print("R2 score:", r2_score_regression)
print("RMSE:", rmse_regression)

R2 score: 0.43942237057756206
RMSE: 54.497938928951996


In [29]:
# Compute the OOB error
def oob_score_regressor(forest, X, y):
    # Initialize an empty list to store OOB predictions for each sample
    n_samples = len(X)
    oob_predictions = [[] for _ in range(n_samples)]

    # For each tree in the forest, get the OOB indices and make predictions for those samples
    for model, oob_indices in forest:
        predictions = model.predict(X[oob_indices])
        for idx, pred in zip(oob_indices, predictions):
            oob_predictions[idx].append(pred)

    # For each sample, find the most common prediction among the trees that did not use it for training
    final_oob_predictions = np.zeros(n_samples)
    for i in range(n_samples):
        # Get the predictions from trees that did not use this sample for training
        tree_preds = np.array(oob_predictions[i])
        if len(tree_preds) > 0: # if there are any OOB predictions for this sample
            final_oob_predictions[i] = np.mean(tree_preds) # mean vote

    # Calculate OOB r2_score
    oob_score = r2_score(y, final_oob_predictions)
    return oob_score

In [30]:
oob_r2_score = oob_score_regressor(fit_regressor(Xr_train, yr_train), Xr_train, yr_train)
print("OOB R2 Score:", oob_r2_score)

OOB R2 Score: 0.38271600822962637


In [36]:
def feature_importance_regression(forest, n_features):
    # Initialize an array to store feature importance scores
    importance_scores = np.zeros(n_features)

    # Recursive function to traverse the tree and update importance scores
    def _traverse(node):
        if node.is_leaf:  # leaf, we stop traversing
            return
        importance_scores[node.feature] += 1
        _traverse(node.left)
        _traverse(node.right)

    # For each tree in the forest, check if the node is a leaf node or not, if not, get the feature index used for splitting and update the importance score for that feature
    for model, _ in forest:
        # Check if the node is a leaf node
        _traverse(model.root)

    # Normalize importance scores
    importance_scores /= np.sum(importance_scores)

    return importance_scores

In [37]:
features_regression = load_diabetes().feature_names
importances_regression = feature_importance_regression(fit_regressor(Xr_train, yr_train), Xr_train.shape[1])

sorted_idx_regression = np.argsort(importances_regression)[::-1]
for i in sorted_idx_regression:
    print(f"{features_regression[i]}: {importances_regression[i]:.4f}")

bmi: 0.2286
s5: 0.2143
s3: 0.1286
bp: 0.1286
s6: 0.1143
s4: 0.0571
s2: 0.0429
s1: 0.0429
age: 0.0286
sex: 0.0143
